In [1]:
# --- repo bootstrap: make src/ importable and run from repo root (works wherever the kernel starts) ---
import sys, os
from pathlib import Path
_ROOT = Path.cwd()
while _ROOT != _ROOT.parent and not (_ROOT / 'src').is_dir():
    _ROOT = _ROOT.parent
sys.path.insert(0, str(_ROOT / 'src'))
os.chdir(_ROOT)

In [2]:
# Cell 1 — one Dune API key per account
from dune_fetch import load_api_keys

API_KEYS, KEY_SOURCE = load_api_keys()


  key 1: loaded from DUNE_API_KEY_1
  key 2: loaded from DUNE_API_KEY_2
  key 3: loaded from DUNE_API_KEY_3


In [3]:
# Cell 2 — query registry, split by which Dune account owns each query.
# Names double as CSV filename prefixes (data_validation.TABLE_LABELS, context/api.md).

QUERY_IDS_1 = {
    "reserve_state_rates":          7711042,
    "oracle_price_usd_eth_weth_6h": 8403530,   # 2h-grain; keep "_6h" — TABLE_LABELS keys off it
}
QUERY_IDS_2 = {
    "borrow_repay":      7798273,
    "liquidation":         7798339,
    "flashloan":         7798349,
    "user_account":      7798351,
    "collateral_toggle": 7798372,
}
QUERY_IDS_3 = {
    "reserve_config": 7804264,   # keep this name — normalize.ipynb globs reserve_config_*.csv
    "supply_withdraw":   8595322,
}

QUERY_GROUPS = {1: QUERY_IDS_1, 2: QUERY_IDS_2, 3: QUERY_IDS_3}

# Static per-asset decimals — don't change, already fetched, not re-run here.
DISABLED_QUERY_IDS = {
    "decimal_reference_part1":                   7711171,
    "decimal_reference_part2":                   7711265,
    "decimal_reference_part3":                   7711276,
    "decimal_reference_part4":                   7711290,
    "decimal_reference_part5":                   7711298,
    "decimal_reference_part6":                   7711304,
    "decimal_reference_part9_collateral_toggle": 7711307,
}

_all_ids = [qid for grp in QUERY_GROUPS.values() for qid in grp.values()]
assert len(_all_ids) == len(set(_all_ids)), "duplicate query id across groups"

for _g, _grp in QUERY_GROUPS.items():
    print(f"  group {_g}: {len(_grp)} queries -> {', '.join(_grp)}")
print(f"  {len(DISABLED_QUERY_IDS)} disabled (not attempted by any group)")


  group 1: 2 queries -> reserve_state_rates, oracle_price_usd_eth_weth_6h
  group 2: 5 queries -> borrow_repay, liquidation, flashloan, user_account, collateral_toggle
  group 3: 2 queries -> reserve_config, supply_withdraw
  7 disabled (not attempted by any group)


In [4]:
# Cell 3 — run settings. Edit this cell between runs.
from dune_fetch import VALID_BUCKET_HOURS

# Which queries actually declare {{bucket_hours}} in their SQL on Dune. Only these are
# sent it. Dune IGNORES a parameter a query does not declare rather than rejecting it, so
# sending it everywhere would look like it worked while those tables ran — and billed — at
# the width hardcoded in their SQL. Move a name in here as you add the parameter to that
# query in the Dune editor; nothing else in this notebook needs to change.
BUCKET_PARAM_ON_DUNE = {"borrow_repay"}

# Per-table parameter tweaks, handed to run_group() as param_overrides.
PARAM_OVERRIDES = {
    # As-of snapshot: upper bound only, so start_date is not sent.
    # Every query here is sent the bare 'YYYY-MM-DD' — the SQL supplies the
    # DATE '...' wrapper. reserve_config (7804264) 400s on that today because its
    # end_date parameter is typed *Date* on Dune; set that Type to *Text* in the query
    # editor and this works as written.
    "reserve_config": {"start_date": None},
}
# A None value is not sent at all, so this opts each un-parameterised table out by name.
for _grp in QUERY_GROUPS.values():
    for _table in _grp:
        if _table not in BUCKET_PARAM_ON_DUNE:
            PARAM_OVERRIDES.setdefault(_table, {})["bucket_hours"] = None

SETTINGS = dict(
    mode="execute",            # "stored" = read latest, free | "execute" = re-run with window below
    start_date="2026-03-28",   # inclusive
    end_date="2026-03-31",     # exclusive
    bucket_hours=6,            # panel grain, in hours; 24 must be a multiple of it
    dry_run=False,             # True -> print the request only, nothing billed
    max_result_mb=30.0,        # abort the download above this size (compute is already spent)
    performance=None,          # None = let Dune pick; key 1 rejects an explicit tier
    timeout_seconds=480,
    poll_seconds=5,
    param_overrides=PARAM_OVERRIDES,
)

_KNOWN_TABLES = {t for _g in QUERY_GROUPS.values() for t in _g}

assert SETTINGS["mode"] in {"stored", "execute"}, "mode must be 'stored' or 'execute'"
assert SETTINGS["bucket_hours"] in VALID_BUCKET_HOURS, (
    f"bucket_hours must be one of {VALID_BUCKET_HOURS} — 24 has to be a multiple of it, "
    "or query 04's grid drifts out of step with the bucket formula")
assert not (BUCKET_PARAM_ON_DUNE - _KNOWN_TABLES), (
    f"BUCKET_PARAM_ON_DUNE names table(s) in no query group: "
    f"{sorted(BUCKET_PARAM_ON_DUNE - _KNOWN_TABLES)}")
if SETTINGS["mode"] == "execute":
    assert SETTINGS["start_date"] < SETTINGS["end_date"], "start_date is not before end_date"

print(f"mode={SETTINGS['mode']}")
if SETTINGS["mode"] == "execute":
    print(f"  window {SETTINGS['start_date']} -> {SETTINGS['end_date']} (end exclusive)")
    print(f"  dry_run={SETTINGS['dry_run']}  max={SETTINGS['max_result_mb']} MB")
    _sent = sorted(BUCKET_PARAM_ON_DUNE)
    _held = sorted(_KNOWN_TABLES - BUCKET_PARAM_ON_DUNE)
    print(f"  bucket_hours={SETTINGS['bucket_hours']} -> {', '.join(_sent) or '(none)'}")
    print(f"  not parameterised on Dune yet: {', '.join(_held) or '(none)'}")
    if _held and SETTINGS["bucket_hours"] != 2:
        print(f"  WARNING: {len(_held)} table(s) still run at their hardcoded 2h grain, so "
              f"a {SETTINGS['bucket_hours']}h panel will NOT line up on (time_bucket, asset) "
              "until their SQL is parameterised on Dune")

mode=execute
  window 2026-03-28 -> 2026-03-31 (end exclusive)
  dry_run=False  max=30.0 MB
  bucket_hours=6 -> borrow_repay
  not parameterised on Dune yet: collateral_toggle, flashloan, liquidation, oracle_price_usd_eth_weth_6h, reserve_config, reserve_state_rates, supply_withdraw, user_account


In [5]:
# Cell 4 — bind registry + keys + settings into run(). Run once.
from dune_fetch import make_runner
from IPython.display import display

run = make_runner(QUERY_GROUPS, API_KEYS, KEY_SOURCE, show=display, **SETTINGS)

print("run(group, tables=None, **overrides) ready — results land in run.results[group]")


run(group, tables=None, **overrides) ready — results land in run.results[group]


In [6]:
# Cell 5 — Account 1 (DUNE_API_KEY_1): reserve_state_rates, oracle_price_usd_eth_weth_6h
run(1)


group 1 (DUNE_API_KEY_1)  mode=execute  2 quer(y/ies)
  window 2026-03-28 -> 2026-03-31 (end exclusive)  dry_run=False
  bucket_hours=6 -> (none)
    NOT sent to reserve_state_rates, oracle_price_usd_eth_weth_6h — each runs at the width hardcoded in its own SQL

reserve_state_rates (7711042)
        01M1MHDVY33E4F8X9TTJJ2XZYV -> QUERY_STATE_EXECUTING
        01M1MHDVY33E4F8X9TTJJ2XZYV -> QUERY_STATE_COMPLETED


KeyboardInterrupt: 

In [ ]:
# Cell 6 — Account 2 (DUNE_API_KEY_2): supply_withdraw, borrow_repay, liquidation,
#           flashloan, user_account, collateral_toggle
run(2)


group 2 (DUNE_API_KEY_2)  mode=execute  1 quer(y/ies)
  window 2026-03-28 -> 2026-03-31 (end exclusive)  dry_run=False
  bucket_hours=6 -> borrow_repay

borrow_repay (7798273)
        01M1MHE7TJ2WYB9V36FWBWBZPE -> QUERY_STATE_PENDING
        01M1MHE7TJ2WYB9V36FWBWBZPE -> QUERY_STATE_EXECUTING
        01M1MHE7TJ2WYB9V36FWBWBZPE -> QUERY_STATE_FAILED
  FAILED  RuntimeError: Execution 01M1MHE7TJ2WYB9V36FWBWBZPE ended as QUERY_STATE_FAILED: Query execution timed out after 2 minutes [Execution ID: 01M1MHE7TJ2WYB9V36FWBWBZPE]


group 2: 0/1 tables (execute mode)
  failures:
    borrow_repay: 7798273: RuntimeError: Execution 01M1MHE7TJ2WYB9V36FWBWBZPE ended as QUERY_STATE_FAILED: Query execution timed out after 2 minutes [Execution ID: 01M1MHE7TJ2WYB9V36FWBWBZPE]


{'tables': {},
 'metas': [],
 'failures': {'borrow_repay': '7798273: RuntimeError: Execution 01M1MHE7TJ2WYB9V36FWBWBZPE ended as QUERY_STATE_FAILED: Query execution timed out after 2 minutes [Execution ID: 01M1MHE7TJ2WYB9V36FWBWBZPE]'}}

In [ ]:
# Cell 7 — Account 3 (DUNE_API_KEY_3): reserve_config
run(3)
